# IMPROVEMENTS ON THE OG NOTEBOOK. *AI generated

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# =====================================================================
# 1. GENERATE AND PREPARE CORE PORTFOLIO DATASET (WITH TIME-SERIES)
# =====================================================================
np.random.seed(42)

sectors = ['Technology', 'Energy', 'Financials', 'Healthcare', 'Manufacturing']
companies = [f"AlphaCorp {i}" for i in range(1, 31)]

data = {
    'Company': companies,
    'Sector': np.random.choice(sectors, 30),
    'Carbon_Intensity': np.random.uniform(10, 500, 30), 
    'Board_Diversity_Pct': np.random.uniform(15, 60, 30),
    'Supply_Chain_Risk': np.random.uniform(20, 90, 30),
    'Governance_Score': np.random.uniform(40, 95, 30),
    'Controversy_Flags': np.random.choice([0, 1, 2, 3], 30, p=[0.6, 0.25, 0.1, 0.05])
}

df = pd.DataFrame(data)

# Sector realism modifiers
df.loc[df['Sector'] == 'Energy', 'Carbon_Intensity'] *= 1.8
df.loc[df['Sector'] == 'Technology', 'Carbon_Intensity'] *= 0.3

# Core Normalization Engine (Min-Max Scaling 0 to 100)
df['E_Score'] = 100 * (1 - (df['Carbon_Intensity'] - df['Carbon_Intensity'].min()) / (df['Carbon_Intensity'].max() - df['Carbon_Intensity'].min()))
s_diversity = 100 * (df['Board_Diversity_Pct'] - df['Board_Diversity_Pct'].min()) / (df['Board_Diversity_Pct'].max() - df['Board_Diversity_Pct'].min())
s_supply = 100 * (1 - (df['Supply_Chain_Risk'] - df['Supply_Chain_Risk'].min()) / (df['Supply_Chain_Risk'].max() - df['Supply_Chain_Risk'].min()))
df['S_Score'] = (s_diversity * 0.5) + (s_supply * 0.5)
g_base = 100 * (df['Governance_Score'] - df['Governance_Score'].min()) / (df['Governance_Score'].max() - df['Governance_Score'].min())
df['G_Score'] = (g_base - (df['Controversy_Flags'] * 15)).clip(lower=0)

# =====================================================================
# 2. DYNAMIC INDUSTRY MATERIALITY WEIGHTS
# =====================================================================
materiality_weights = {
    'Energy': {'E': 0.60, 'S': 0.20, 'G': 0.20},
    'Manufacturing': {'E': 0.50, 'S': 0.25, 'G': 0.25},
    'Technology': {'E': 0.20, 'S': 0.40, 'G': 0.40},
    'Healthcare': {'E': 0.20, 'S': 0.40, 'G': 0.40},
    'Financials': {'E': 0.10, 'S': 0.40, 'G': 0.50}
}

# Current Quarter (Q4) Score
df['Total_ESG_Score_Q4'] = df.apply(
    lambda row: (row['E_Score'] * materiality_weights[row['Sector']]['E']) +
                (row['S_Score'] * materiality_weights[row['Sector']]['S']) +
                (row['G_Score'] * materiality_weights[row['Sector']]['G']), 
    axis=1
)

# =====================================================================
# 3. ADDITION 1: TIME-SERIES BACK-HISTORY GENERATION & MOMENTUM
# =====================================================================
# Generate simulated historical data for previous quarters (Q1, Q2, Q3) to measure trajectory
df['Total_ESG_Score_Q3'] = df['Total_ESG_Score_Q4'] - np.random.uniform(-4, 5, 30)
df['Total_ESG_Score_Q2'] = df['Total_ESG_Score_Q3'] - np.random.uniform(-3, 6, 30)
df['Total_ESG_Score_Q1'] = df['Total_ESG_Score_Q2'] - np.random.uniform(-5, 5, 30)

# Calculate ESG Momentum (4-Quarter Delta)
df['ESG_Momentum_YTD'] = df['Total_ESG_Score_Q4'] - df['Total_ESG_Score_Q1']

# =====================================================================
# 4. ADDITION 2: PEER PERCENTILE RANKING (SECTOR RELATIVE)
# =====================================================================
# Compute where the company stands relative only to its industry peers (0 = Worst, 100 = Best)
df['Sector_Relative_Percentile'] = df.groupby('Sector')['Total_ESG_Score_Q4'].rank(pct=True) * 100

# =====================================================================
# 5. RISK CLASSIFICATION ENGINE (INTEGRATED METRICS)
# =====================================================================
def classify_investment_action(row):
    score = row['Total_ESG_Score_Q4']
    momentum = row['ESG_Momentum_YTD']
    
    if score < 50:
        if momentum > 3:
            return 'Watchlist (Improving Laggard)'
        return 'Immediate Divest (Structural Laggard)'
    elif score >= 75:
        if momentum < -3:
            return 'Trim Position (Declining Leader)'
        return 'Strong Hold (Core Leader)'
    else:
        return 'Neutral Allocation'

df['Asset_Management_Action'] = df.apply(classify_investment_action, axis=1)

# Statistical Outliers & OLS Integration (Using latest Q4 Data)
df['Carbon_Z'] = (df['Carbon_Intensity'] - df['Carbon_Intensity'].mean()) / df['Carbon_Intensity'].std()
df['Supply_Z'] = (df['Supply_Chain_Risk'] - df['Supply_Chain_Risk'].mean()) / df['Supply_Chain_Risk'].std()

X = df[['Carbon_Intensity', 'Board_Diversity_Pct', 'Supply_Chain_Risk', 'Controversy_Flags']]
X = sm.add_constant(X)
ols_model = sm.OLS(df['Total_ESG_Score_Q4'], X).fit()

# Export updated data
df.to_csv('esg_portfolio_scorecard.csv', index=False)

print("="*60)
print("ADVANCED QUARTERLY MOMENTUM & PEER RANKINGS GENERATED")
print(df[['Company', 'Sector', 'Total_ESG_Score_Q4', 'Sector_Relative_Percentile', 'ESG_Momentum_YTD', 'Asset_Management_Action']].head(5))
print("="*60)